# Extraction Tunisie Clearing - spreads corporate 2023-2025

Objectif : produire un jeu de donnees auditable des primes corporate publiees dans les bulletins Tunisie Clearing. Le notebook reste volontairement court; la logique reutilisable est dans `src/data/tunisie_clearing/`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import plotly.express as px

from data.tunisie_clearing.build_corporate_spreads_dataset import run_full_pipeline, build_corporate_spreads_dataset
from data.tunisie_clearing.common import resolve_paths

paths = resolve_paths(PROJECT_ROOT)
pd.set_option("display.max_colwidth", 160)

## Execution du pipeline

La cellule suivante scrape l'historique, telecharge les PDF, extrait le texte et construit les fichiers finaux. Si le site est temporairement indisponible, relancer uniquement les etapes necessaires depuis les modules.

In [2]:
RUN_FULL_PIPELINE = True

if RUN_FULL_PIPELINE:
    clean = run_full_pipeline(project_root=PROJECT_ROOT)
else:
    clean = build_corporate_spreads_dataset(project_root=PROJECT_ROOT)

clean.head()

,date,year,month,sector,spread_type,spread_percent,spread_decimal,previous_spread_percent,final_spread_percent,source_pdf,source_url,source_text_snippet,extraction_method,extraction_confidence,data_quality_flag,extraction_status
0,NaN,2023,NaN,,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NO_OFFICIAL_CORPORATE_CURVE_BEFORE_END_2023,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED
1,NaN,2024,NaN,,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NO_CORPORATE_SPREAD_SECTION_DETECTED,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED_IN_YEAR
2,NaN,2025,NaN,,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NO_CORPORATE_SPREAD_SECTION_DETECTED,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED_IN_YEAR


## Bulletins trouves

In [3]:
bulletins = pd.read_csv(paths.bulletins_index_csv) if paths.bulletins_index_csv.exists() else pd.DataFrame()
bulletins[["bulletin_date", "year", "month", "pdf_filename", "download_status", "pdf_url"]].head(20) if not bulletins.empty else bulletins

,bulletin_date,year,month,pdf_filename,download_status,pdf_url
0,2023-04-24,2023,4,04c493ae-b3a4-423e-bbb6-2eab3caa2211.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202304/04c493ae-b3a4-423e-bbb6-2eab3caa2211.pdf
1,2023-04-26,2023,4,04f097ff-744d-4426-9a0e-0c386a29413c.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202304/04f097ff-744d-4426-9a0e-0c386a29413c.pdf
2,2023-04-20,2023,4,0a8cfa04-749e-48ec-9e74-5832f5ae683d.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202407/0a8cfa04-749e-48ec-9e74-5832f5ae683d.pdf
3,2023-04-27,2023,4,22a258a0-0ac6-48d8-896c-3396f676668c.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202304/22a258a0-0ac6-48d8-896c-3396f676668c.pdf
4,2023-04-28,2023,4,4bc02e9a-7864-4f61-8b16-2d4762e95cbd.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202305/4bc02e9a-7864-4f61-8b16-2d4762e95cbd.pdf
5,2023-04-20,2023,4,5ad56ff8-b3eb-440d-b570-30cabb5ee3da.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202304/5ad56ff8-b3eb-440d-b570-30cabb5ee3da.pdf
6,2023-04-25,2023,4,eb0df679-9ebd-4a84-9901-eeea04d651c3.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202304/eb0df679-9ebd-4a84-9901-eeea04d651c3.pdf
7,2023-04-19,2023,4,f9e25430-1843-4280-bf68-681e5ac4e76d.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202304/f9e25430-1843-4280-bf68-681e5ac4e76d.pdf
8,2023-05-17,2023,5,0d2ba892-b1ff-4660-b6c6-e4e2d14e66ae.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202305/0d2ba892-b1ff-4660-b6c6-e4e2d14e66ae.pdf
9,2023-05-22,2023,5,1929d0cb-6211-4929-8166-26a1fdf5a7bc.pdf,ALREADY_EXISTS,https://www.tunisieclearing.com/upload/202305/1929d0cb-6211-4929-8166-26a1fdf5a7bc.pdf


## Spreads extraits

In [4]:
cols = ["date", "year", "month", "sector", "spread_type", "spread_percent", "previous_spread_percent", "final_spread_percent", "extraction_confidence", "data_quality_flag"]
clean[cols].sort_values(["year", "month", "sector"]).head(50) if not clean.empty else clean

,date,year,month,sector,spread_type,spread_percent,previous_spread_percent,final_spread_percent,extraction_confidence,data_quality_flag
0,NaN,2023,NaN,,NaN,None,None,None,NaN,NO_OFFICIAL_CORPORATE_CURVE_BEFORE_END_2023
1,NaN,2024,NaN,,NaN,None,None,None,NaN,NO_CORPORATE_SPREAD_SECTION_DETECTED
2,NaN,2025,NaN,,NaN,None,None,None,NaN,NO_CORPORATE_SPREAD_SECTION_DETECTED


## Synthese annuelle

In [5]:
if not clean.empty:
    annual = (
        clean.groupby(["year", "sector", "extraction_status"], dropna=False)
        .size()
        .reset_index(name="observations")
        .sort_values(["year", "sector"])
    )
else:
    annual = pd.DataFrame(columns=["year", "sector", "extraction_status", "observations"])
annual

,year,sector,extraction_status,observations
0,2023,,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED,1
1,2024,,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED_IN_YEAR,1
2,2025,,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED_IN_YEAR,1


## Courbes temporelles

In [6]:
plot_df = clean[clean["extraction_status"].eq("EXTRACTED")].copy() if not clean.empty else pd.DataFrame()
if not plot_df.empty:
    plot_df["date"] = pd.to_datetime(plot_df["date"], errors="coerce")
    px.line(plot_df.sort_values("date"), x="date", y="spread_percent", color="sector", markers=True, title="Evolution des primes corporate par secteur")
else:
    print("Aucune observation exploitable pour le graphique temporel.")

Aucune observation exploitable pour le graphique temporel.


### Graphiques demandes

In [7]:
def line_for_sector(frame, sector, years, title):
    subset = frame[(frame["sector"].eq(sector)) & (frame["year"].isin(years))].copy()
    if subset.empty:
        print(f"Aucune donnee pour {title}.")
        return None
    subset["date"] = pd.to_datetime(subset["date"], errors="coerce")
    return px.line(subset.sort_values("date"), x="date", y="spread_percent", markers=True, title=title)

fig_bancaire = line_for_sector(plot_df, "BANCAIRE", [2025], "Evolution du spread bancaire 2025")
fig_bancaire

Aucune donnee pour Evolution du spread bancaire 2025.


In [8]:
fig_leasing = line_for_sector(plot_df, "LEASING", [2024, 2025], "Evolution du spread leasing 2024-2025")
fig_leasing

Aucune donnee pour Evolution du spread leasing 2024-2025.


In [9]:
fig_microfinance = line_for_sector(plot_df, "MICROFINANCE", [2025], "Evolution du spread microfinance 2025")
fig_microfinance

Aucune donnee pour Evolution du spread microfinance 2025.


In [10]:
text_index = pd.read_csv(paths.extracted_text_index_csv) if paths.extracted_text_index_csv.exists() else pd.DataFrame()
if not text_index.empty:
    monthly = text_index[text_index["contains_corporate_section"].astype(bool)].groupby(["year", "month"]).size().reset_index(name="bulletins")
    monthly["periode"] = pd.to_datetime(monthly["year"].astype(str) + "-" + monthly["month"].astype(str) + "-01")
    px.bar(monthly, x="periode", y="bulletins", title="Nombre de bulletins avec donnees corporate par mois")
else:
    print("Index d'extraction indisponible.")

In [11]:
if not plot_df.empty:
    availability = plot_df.assign(value=1).pivot_table(index="sector", columns=["year", "month"], values="value", aggfunc="max", fill_value=0)
    availability.columns = [f"{year}-{month:02d}" for year, month in availability.columns]
    availability_long = availability.reset_index().melt(id_vars="sector", var_name="mois", value_name="disponible")
    px.density_heatmap(availability_long, x="mois", y="sector", z="disponible", title="Heatmap de disponibilite des spreads par secteur et par mois")
else:
    print("Aucune disponibilite a afficher.")

Aucune disponibilite a afficher.


## Observations manquantes ou ambigues

In [12]:
missing = clean[
    clean["extraction_status"].astype(str).str.contains("AMBIG|NO_OFFICIAL|MISSING|FAILED", case=False, na=False)
    | clean["data_quality_flag"].astype(str).str.contains("AMBIG|NO_OFFICIAL|LIMITED|MISSING", case=False, na=False)
].copy() if not clean.empty else pd.DataFrame()
missing

,date,year,month,sector,spread_type,spread_percent,spread_decimal,previous_spread_percent,final_spread_percent,source_pdf,source_url,source_text_snippet,extraction_method,extraction_confidence,data_quality_flag,extraction_status
0,NaN,2023,NaN,,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NO_OFFICIAL_CORPORATE_CURVE_BEFORE_END_2023,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED
1,NaN,2024,NaN,,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NO_CORPORATE_SPREAD_SECTION_DETECTED,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED_IN_YEAR
2,NaN,2025,NaN,,NaN,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NO_CORPORATE_SPREAD_SECTION_DETECTED,NO_OFFICIAL_CORPORATE_SPREAD_DETECTED_IN_YEAR


## Limites de couverture

- 2023 : aucune prime n'est inventee. Si aucune section officielle n'est detectee, la sortie porte le flag `NO_OFFICIAL_CORPORATE_CURVE_BEFORE_END_2023`.
- 2024 : les bulletins peuvent decrire surtout le secteur leasing et citer une prime ancienne puis une prime finale.
- 2025 : la couverture attendue est plus large, mais chaque ligne conserve l'extrait source pour audit.